# 03 - transactions aggregation

same thing as last time but for `transactions_v2.csv`.

each user can show up multiple times here (one row per transaction), but `train.csv` only has one row per user. so just like with user_logs, we need to squish this down to one row per user.

this file's only ~115MB though, so pandas is fine - don't need polars for this one.


## Step 1: load the file

basically the same setup as before.

- `pd`, `Path` - same as notebook 01
- `DATA` - raw csv folder
- `PROCESSED` - output folder, same one from last notebook
- this file's small enough to just `read_csv` the whole thing, no need to be fancy about it


In [1]:
import pandas as pd
from pathlib import Path

DATA = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

transactions = pd.read_csv(DATA / "transactions_v2.csv")
print(transactions.shape)
transactions.head()


(1431009, 9)


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,++6eU4LsQ3UQ20ILS7d99XK8WbiVgbyYL4FUgzZR134=,32,90,298,298,0,20170131,20170504,0
1,++lvGPJOinuin/8esghpnqdljm6NXS8m8Zwchc7gOeA=,41,30,149,149,1,20150809,20190412,0
2,+/GXNtXWQVfKrEDqYAzcSw2xSPYMKWNj22m+5XkVQZc=,36,30,180,180,1,20170303,20170422,0
3,+/w1UrZwyka4C9oNH3+Q8fUf3fD8R3EwWrx57ODIsqk=,36,30,180,180,1,20170329,20170331,1
4,+00PGzKTYqtnb65mPKPyeHXcZEwqiEzktpQksaaSC3c=,41,30,99,99,1,20170323,20170423,0


## Step 2: do users have more than one transaction?

before we aggregate, just want to check - is it one transaction per user, or do some people show up a bunch of times?

- `value_counts()` on `msno` - counts how many rows each user has
- `.describe()` - quick min/max/avg so we get a feel for what we're dealing with


In [2]:
counts = transactions['msno'].value_counts()
print(counts.describe())
counts.head()


count    1.197050e+06
mean     1.195446e+00
std      1.206252e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      2.080000e+02
Name: count, dtype: float64


msno
72gJqt1O31E/WoxAEYFn9LHNI6mAZFGera5Q6gvsFkA=    208
5ty4nZkq54z93wQtBN7RHVYj8rNghBDCVBH+3xmxf0I=    172
OGKDrZQDB3yewZhoSd5qqvmG5A1GcNTYMexO95NlH+g=    148
WHsCtkOVsauvqBL0ULuG38887y7aU8GXdCmJMjw6hjQ=    145
SNlFRAsmUqnXKPofSXA8WYUc5DtmLcUMy4pXSJ3Ohz0=    131
Name: count, dtype: int64

## Step 3: fix the dates

same thing as notebook 01 - `transaction_date` and `membership_expire_date` are just numbers like `20170131`, so let's turn them into actual dates.


In [3]:
for col in ['transaction_date', 'membership_expire_date']:
    transactions[col] = pd.to_datetime(transactions[col], format='%Y%m%d', errors='coerce')

transactions[['transaction_date', 'membership_expire_date']].describe()


,transaction_date,membership_expire_date
count,1431009,1431009
mean,2017-01-25 20:15:32.129986560,2017-05-24 09:47:08.276551168
min,2015-01-01 00:00:00,2016-04-19 00:00:00
25%,2017-02-28 00:00:00,2017-04-10 00:00:00
50%,2017-03-11 00:00:00,2017-04-21 00:00:00
75%,2017-03-23 00:00:00,2017-05-01 00:00:00
max,2017-03-31 00:00:00,2036-10-15 00:00:00


## Step 4: group by user

now the actual aggregation - one row per user instead of one row per transaction.

- `groupby('msno')` - bunch up all the rows for each user
- `.agg(...)` - for each user, work out:
  - `num_transactions` - how many transactions they made (just a row count)
  - `total_actual_paid` - how much they've paid in total
  - `total_plan_list_price` - same idea but for the list price
  - `is_cancel_sum` - how many times they hit cancel
  - `is_auto_renew_mean` - what fraction of their transactions had auto-renew turned on
  - `last_transaction_date` - their most recent transaction
  - `last_expire_date` - the latest membership expiry date we have for them
- writing it as `new_name=('column', 'function')` lets us rename the columns while we aggregate, so we don't end up with messy default names
- `.reset_index()` - after groupby, `msno` becomes the index. this just puts it back as a normal column


In [4]:
user_transactions = (
    transactions
    .groupby('msno')
    .agg(
        num_transactions=('msno', 'count'),
        total_actual_paid=('actual_amount_paid', 'sum'),
        total_plan_list_price=('plan_list_price', 'sum'),
        is_cancel_sum=('is_cancel', 'sum'),
        is_auto_renew_mean=('is_auto_renew', 'mean'),
        last_transaction_date=('transaction_date', 'max'),
        last_expire_date=('membership_expire_date', 'max'),
    )
    .reset_index()
)


## Step 5: did it work?

- `.shape` - should be way fewer rows now, roughly one per unique user (~1.2 million from notebook 01)
- `.head()` - eyeball the first few rows, make sure the numbers look reasonable


In [5]:
print(user_transactions.shape)
user_transactions.head()


(1197050, 8)


,msno,num_transactions,total_actual_paid,total_plan_list_price,is_cancel_sum,is_auto_renew_mean,last_transaction_date,last_expire_date
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,1,1599,1599,0,0.0,2016-10-23,2018-02-06
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,1,99,99,0,1.0,2017-03-15,2017-04-15
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,2,298,298,0,1.0,2017-03-31,2017-05-19
3,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,1,149,149,0,1.0,2017-03-26,2017-04-26
4,++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk=,1,149,149,0,1.0,2017-03-15,2017-04-15


## Step 6: save it

same as before - save the result so we're not redoing this every time we open the notebook.


In [6]:
user_transactions.to_parquet(PROCESSED / "transactions_v2_agg.parquet")
print("saved:", PROCESSED / "transactions_v2_agg.parquet")


saved: ../data/processed/transactions_v2_agg.parquet
